# Финальный проект по Process Mining

Цель: взять event log из `final_dataset.csv`, разделить процессы на группы, оценить качество кластеризации и объяснить, какие варианты процесса есть в данных.

Перед запуском один раз установите зависимости:

```bash
pip install -r requirements.txt
```

In [ ]:
from pathlib import Path
from collections import Counter
import warnings

import numpy as np
import pandas as pd
from scipy import sparse

from IPython.display import display, Markdown

import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from hdbscan.validity import validity_index

import pm4py
from pm4py.algo.discovery.alpha import algorithm as alpha_miner
from pm4py.algo.evaluation.replay_fitness import algorithm as replay_fitness
from pm4py.visualization.petri_net import visualizer as pn_visualizer

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

DATA_PATH = Path("final_dataset.csv")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

CASE = "case concept:name"
ACTIVITY = "event concept:name"
TIME = "event time:timestamp"
VALUE = "event Cumulative net worth (EUR)"
RANDOM_STATE = 42

## 1. Читаем event log

В process mining важны три колонки:

- `case concept:name` — id одного процесса;
- `event concept:name` — событие внутри процесса;
- `event time:timestamp` — время события.

In [ ]:
df = pd.read_csv(DATA_PATH)
df[TIME] = pd.to_datetime(df[TIME], dayfirst=True, errors="coerce")
df = df.dropna(subset=[CASE, ACTIVITY, TIME]).copy()
df = df.sort_values([CASE, TIME]).reset_index(drop=True)

df[CASE] = df[CASE].astype(str)
df[ACTIVITY] = df[ACTIVITY].astype(str)

print("events:", len(df))
print("cases:", df[CASE].nunique())
print("activities:", df[ACTIVITY].nunique())
print("period:", df[TIME].min(), "—", df[TIME].max())
display(df.head())

## 2. Быстрый обзор процесса

Смотрим самые частые события и длительность cases. Это базовая картина процесса до кластеризации.

In [ ]:
case_start_end = df.groupby(CASE)[TIME].agg(start="min", end="max")
case_duration = ((case_start_end["end"] - case_start_end["start"]).dt.total_seconds() / 86400).rename("duration_days")
case_event_count = df.groupby(CASE).size().rename("event_count")

top_activities = df[ACTIVITY].value_counts().head(15).reset_index()
top_activities.columns = ["activity", "events"]

display(top_activities)
display(case_duration.describe())

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(data=top_activities, x="events", y="activity", ax=axes[0], color="#3a78a1")
axes[0].set_title("Top-15 событий")
axes[0].set_ylabel("")

sns.histplot(case_duration.clip(upper=case_duration.quantile(0.99)), bins=40, ax=axes[1], color="#9a6a2f")
axes[1].set_title("Длительность case, дни")
axes[1].set_xlabel("days")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "eda_activity_duration.png", dpi=160, bbox_inches="tight")
plt.show()

## 3. Собираем признаки на уровне case

Кластеризовать нужно не строки CSV, а целые процессы. Поэтому для каждого `case` собираем признаки: длительность, число событий, частоты activity, категориальные поля и текстовые описания.

In [ ]:
case_features = pd.DataFrame(index=case_event_count.index)
case_features["event_count"] = case_event_count
case_features["duration_days"] = case_duration
case_features["unique_activities"] = df.groupby(CASE)[ACTIVITY].nunique()
case_features["value_mean"] = df.groupby(CASE)[VALUE].mean() if VALUE in df.columns else 0
case_features = case_features.fillna(0)

activity_features = pd.crosstab(df[CASE], df[ACTIVITY])
activity_features = activity_features.div(activity_features.sum(axis=1), axis=0).fillna(0)
activity_features.columns = ["activity__" + str(c) for c in activity_features.columns]

cat_cols = [
    "case Spend area text",
    "case Company",
    "case Document Type",
    "case Vendor",
    "case Item Type",
    "case Item Category",
    "case Spend classification text",
    "case Source",
]
cat_cols = [c for c in cat_cols if c in df.columns]
case_categories = df.groupby(CASE)[cat_cols].first().reindex(case_features.index).fillna("UNKNOWN").astype(str)

text_cols = ["case Spend area text", "case Sub spend area text", "case Spend classification text", "case Name"]
text_cols = [c for c in text_cols if c in df.columns]
case_text = df.groupby(CASE)[text_cols].first().reindex(case_features.index).fillna("").astype(str)
case_text_joined = case_text.agg(" ".join, axis=1)

encoder = OneHotEncoder(handle_unknown="ignore", min_frequency=0.01, sparse_output=True)
X_cat = encoder.fit_transform(case_categories)

vectorizer = TfidfVectorizer(max_features=250, min_df=5, stop_words="english", ngram_range=(1, 2))
X_text = vectorizer.fit_transform(case_text_joined)

scaler = StandardScaler()
X_num = sparse.csr_matrix(scaler.fit_transform(case_features))
X_act = sparse.csr_matrix(activity_features.reindex(case_features.index).fillna(0).to_numpy())

X = sparse.hstack([X_num, X_act, X_cat, X_text]).tocsr()

print("cases x features:", X.shape)
display(case_features.head())

## 4. Делим процессы на кластеры

Каждая строка теперь описывает один case. Сначала сжимаем признаки через SVD, потом запускаем KMeans и выбираем число кластеров по Silhouette. DBSCAN добавлен как второй вариант кластеризации.

In [ ]:
svd = TruncatedSVD(n_components=30, random_state=RANDOM_STATE)
X_reduced = svd.fit_transform(X)
X_scaled = StandardScaler().fit_transform(X_reduced)

sample_size = min(10000, len(X_scaled))
sample_idx = np.random.default_rng(RANDOM_STATE).choice(len(X_scaled), sample_size, replace=False)

scores = []
for k in range(2, 9):
    labels = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE).fit_predict(X_scaled)
    score = silhouette_score(X_scaled[sample_idx], labels[sample_idx])
    scores.append((k, score, labels))

best_k, best_silhouette, kmeans_labels = max(scores, key=lambda x: x[1])
case_features["cluster_kmeans"] = kmeans_labels

score_table = pd.DataFrame(scores, columns=["k", "silhouette", "labels"]).drop(columns="labels")
display(score_table)
print("best k:", best_k)
print("best silhouette:", round(best_silhouette, 4))

fig, ax = plt.subplots(figsize=(7, 4))
sns.lineplot(data=score_table, x="k", y="silhouette", marker="o", ax=ax)
ax.set_title("Выбор k для KMeans")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "kmeans_silhouette_selection.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
neighbors = NearestNeighbors(n_neighbors=25).fit(X_scaled)
distances, _ = neighbors.kneighbors(X_scaled)
eps = np.quantile(distances[:, -1], 0.85)

dbscan_labels = DBSCAN(eps=eps, min_samples=25, n_jobs=-1).fit_predict(X_scaled)
case_features["cluster_dbscan"] = dbscan_labels

print("DBSCAN eps:", round(eps, 4))
print("DBSCAN clusters:", len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0))
print("DBSCAN noise share:", round((dbscan_labels == -1).mean(), 3))

## 5. Метрики качества кластеризации

- `Silhouette` показывает, насколько хорошо группы отделены друг от друга.
- `DBCV` оценивает density-based структуру кластеров.
- `Fitness` ниже будет считаться уже для process model через pm4py.

In [ ]:
def dbcv_score(X_data, labels):
    labels = np.asarray(labels)
    keep = labels != -1
    if len(set(labels[keep])) < 2:
        return np.nan
    idx = np.where(keep)[0]
    idx = np.random.default_rng(RANDOM_STATE).choice(idx, min(5000, len(idx)), replace=False)
    try:
        return validity_index(X_data[idx].astype(float), labels[idx].astype(int), metric="euclidean")
    except ValueError:
        return np.nan

quality = pd.DataFrame([
    {
        "algorithm": "KMeans",
        "clusters": len(set(kmeans_labels)),
        "silhouette": silhouette_score(X_scaled[sample_idx], kmeans_labels[sample_idx]),
        "dbcv": dbcv_score(X_scaled, kmeans_labels),
    },
    {
        "algorithm": "DBSCAN",
        "clusters": len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0),
        "silhouette": silhouette_score(X_scaled[sample_idx], dbscan_labels[sample_idx]) if len(set(dbscan_labels)) > 1 else np.nan,
        "dbcv": dbcv_score(X_scaled, dbscan_labels),
    },
])

display(quality)

## 6. DFG: граф процесса

DFG показывает, какие события чаще всего идут друг за другом. Это самая простая визуализация процесса.

In [ ]:
df_clusters = df.merge(case_features[["cluster_kmeans"]], left_on=CASE, right_index=True, how="left")

def plot_dfg(data, title, file_name, top_edges=40):
    edges = Counter()
    for trace in data.groupby(CASE)[ACTIVITY].agg(list):
        for a, b in zip(trace, trace[1:]):
            edges[(a, b)] += 1

    graph = nx.DiGraph()
    for (a, b), weight in edges.most_common(top_edges):
        graph.add_edge(a, b, weight=weight)

    plt.figure(figsize=(15, 9))
    pos = nx.spring_layout(graph, seed=RANDOM_STATE, k=1.3)
    weights = [graph[a][b]["weight"] for a, b in graph.edges]
    max_weight = max(weights) if weights else 1
    widths = [1 + 4 * w / max_weight for w in weights]

    nx.draw_networkx_nodes(graph, pos, node_size=900, node_color="#d9ecf7", edgecolors="#315f7d")
    nx.draw_networkx_edges(graph, pos, width=widths, arrows=True, arrowstyle="-|>", arrowsize=15, edge_color="#777777")
    nx.draw_networkx_labels(graph, pos, font_size=8)
    nx.draw_networkx_edge_labels(graph, pos, edge_labels={(a, b): graph[a][b]["weight"] for a, b in graph.edges}, font_size=7)
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / file_name, dpi=160, bbox_inches="tight")
    plt.show()

plot_dfg(df_clusters, "DFG всего процесса", "dfg_full_process.png")

biggest_cluster = case_features["cluster_kmeans"].value_counts().idxmax()
plot_dfg(
    df_clusters[df_clusters["cluster_kmeans"] == biggest_cluster],
    f"DFG самого большого кластера: {biggest_cluster}",
    f"dfg_biggest_cluster_{biggest_cluster}.png",
)

## 7. Petri net и Fitness

Строим сеть Петри через Alpha miner. Для скорости берём фиксированную подвыборку cases.

In [ ]:
all_cases = df[CASE].drop_duplicates().to_numpy()
sample_cases = np.random.default_rng(RANDOM_STATE).choice(all_cases, min(5000, len(all_cases)), replace=False)
pm_df = df[df[CASE].isin(sample_cases)][[CASE, ACTIVITY, TIME]].copy()

pm_df = pm4py.format_dataframe(pm_df, case_id=CASE, activity_key=ACTIVITY, timestamp_key=TIME)
event_log = pm4py.convert_to_event_log(pm_df)

net, initial_marking, final_marking = alpha_miner.apply(event_log)
fitness = replay_fitness.apply(event_log, net, initial_marking, final_marking, variant=replay_fitness.Variants.TOKEN_BASED)

display(pd.DataFrame([fitness]))

petri_picture = pn_visualizer.apply(net, initial_marking, final_marking)
pn_visualizer.save(petri_picture, str(OUTPUT_DIR / "petri_net_alpha.png"))
display(petri_picture)

## 8. Метрики по кластерам и частые циклы

Теперь можно понять, какие кластеры большие, какие долгие и где есть повторная работа.

In [ ]:
cluster_summary = (
    case_features.groupby("cluster_kmeans")
    .agg(
        cases=("event_count", "size"),
        avg_events=("event_count", "mean"),
        avg_duration_days=("duration_days", "mean"),
        median_duration_days=("duration_days", "median"),
        avg_value=("value_mean", "mean"),
    )
    .sort_values("cases", ascending=False)
)
cluster_summary["case_share"] = cluster_summary["cases"] / cluster_summary["cases"].sum()
display(cluster_summary)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
cluster_summary["cases"].plot(kind="bar", ax=axes[0], color="#3a78a1", title="Размер кластеров")
cluster_summary["avg_duration_days"].plot(kind="bar", ax=axes[1], color="#9a6a2f", title="Средняя длительность")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "cluster_process_metrics.png", dpi=160, bbox_inches="tight")
plt.show()

repeat_counter = Counter()
cycle_counter = Counter()
for trace in df.groupby(CASE)[ACTIVITY].agg(list):
    counts = Counter(trace)
    for activity, count in counts.items():
        if count > 1:
            repeat_counter[activity] += count - 1

    last_seen = {}
    for i, activity in enumerate(trace):
        if activity in last_seen and i - last_seen[activity] > 1:
            cycle_counter[f"{activity} -> ... -> {activity}"] += 1
        last_seen[activity] = i

repeated_activities = pd.DataFrame(repeat_counter.most_common(10), columns=["activity", "extra_repetitions"])
frequent_cycles = pd.DataFrame(cycle_counter.most_common(10), columns=["cycle", "count"])

display(repeated_activities)
display(frequent_cycles)

## 9. Какие признаки влияют на длительность процесса

Обучаем простую модель: по признакам case предсказываем длительность. Самые важные признаки интерпретируем как факторы, связанные со временем выполнения процесса.

In [ ]:
model_features = sparse.hstack([X_act, X_cat, X_text]).tocsr()
feature_names = (
    list(activity_features.columns)
    + list(encoder.get_feature_names_out(cat_cols))
    + ["text__" + t for t in vectorizer.get_feature_names_out()]
)

y = case_features["duration_days"].clip(lower=0)
X_train, X_test, y_train, y_test = train_test_split(model_features, y, test_size=0.2, random_state=RANDOM_STATE)

rf = RandomForestRegressor(n_estimators=150, max_depth=12, min_samples_leaf=10, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)

importance = pd.DataFrame({"feature": feature_names, "importance": rf.feature_importances_})
importance = importance.sort_values("importance", ascending=False).head(20)

print("R2 train:", round(rf.score(X_train, y_train), 3))
print("R2 test:", round(rf.score(X_test, y_test), 3))
display(importance)

plt.figure(figsize=(10, 7))
sns.barplot(data=importance.head(15), x="importance", y="feature", color="#4f8f6f")
plt.title("Feature importance для длительности процесса")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "duration_feature_importance.png", dpi=160, bbox_inches="tight")
plt.show()

## 10. Кластеризация текста

Отдельно кластеризуем текстовые описания: spend area, sub spend area, classification и vendor name. Смысл текстового кластера объясняем через top terms.

In [ ]:
text_vectorizer = TfidfVectorizer(max_features=500, min_df=3, stop_words="english", ngram_range=(1, 2))
X_text_only = text_vectorizer.fit_transform(case_text_joined)

text_k = 6
text_model = KMeans(n_clusters=text_k, n_init=20, random_state=RANDOM_STATE)
text_labels = text_model.fit_predict(X_text_only)
case_features["cluster_text"] = text_labels

terms = np.array(text_vectorizer.get_feature_names_out())
centers = text_model.cluster_centers_

rows = []
for cluster in range(text_k):
    top_terms = terms[np.argsort(centers[cluster])[-10:][::-1]]
    rows.append({
        "text_cluster": cluster,
        "cases": int((text_labels == cluster).sum()),
        "top_terms": ", ".join(top_terms),
    })

text_summary = pd.DataFrame(rows).sort_values("cases", ascending=False)
display(text_summary)

## 11. Итоговые выводы

Ниже автоматически собирается короткий текст для отчета и защиты.

In [ ]:
fitness_value = fitness.get("log_fitness", fitness.get("average_trace_fitness", np.nan))
slowest_cluster = cluster_summary["avg_duration_days"].idxmax()
biggest_cluster = cluster_summary["cases"].idxmax()
main_cycle = frequent_cycles.iloc[0]["cycle"] if len(frequent_cycles) else "не найден"
main_cycle_count = int(frequent_cycles.iloc[0]["count"]) if len(frequent_cycles) else 0

text = f"""
### Выводы

1. В данных {len(df):,} событий, {df[CASE].nunique():,} cases и {df[ACTIVITY].nunique():,} типов событий.

2. Процессы были разделены на {best_k} KMeans-кластера. Silhouette = {best_silhouette:.3f}. Это умеренное качество: процесс закупок вариативный, поэтому группы пересекаются.

3. Самый большой кластер: {biggest_cluster}. Самый долгий по средней длительности: {slowest_cluster}.

4. Fitness сети Петри через Alpha miner = {fitness_value:.3f}. Значение ниже 1 нормально для реального процесса: есть редкие ветки, отмены, изменения и циклы.

5. Самый частый цикл: `{main_cycle}`, количество повторов: {main_cycle_count}. Это признак rework — процесс возвращается к уже встречавшемуся действию.

6. Feature importance показывает, какие activity, категории и текстовые признаки сильнее связаны с длительностью процесса.

7. Текстовая кластеризация показывает тематические группы закупок по spend area, classification и vendor name.
"""

display(Markdown(text))

## Что говорить на защите

Коротко:

> Я рассматривал датасет как event log. Один `case concept:name` — это один процесс, а `event concept:name` — события внутри него. Сначала я собрал признаки на уровне case, потом разделил cases на кластеры. После этого сравнил кластеры по длительности, построил DFG и сеть Петри, посчитал fitness, нашел частые циклы и факторы, влияющие на длительность процесса.

Главная идея: найти разные варианты прохождения процесса и объяснить, какие из них быстрее, медленнее или проблемнее.